## Pratice for Context Management in OPENAI Agents SDK

In [1]:
from agents import Agent, Runner, function_tool
from agents import RunContextWrapper, TResponseInputItem
from dataclasses import dataclass
import random
import time
import asyncio

In [6]:
@dataclass
class UserProfile:
    id: int
    name: str
    shopping_cart: list[str]


@function_tool
def get_budget(wrapper: RunContextWrapper[UserProfile]):
    """Get the account Balance of the user using the user's id and thier linked bank account"""
    print("Getting Account Balance\n")
    time.sleep(0.5)
    user_id = wrapper.context.id

    # pretend we are fetching user's current balance from account
    return 100.0


@function_tool
async def search_for_item(wrapper: RunContextWrapper[UserProfile], item: str) -> str:
    """Search for item in the database"""
    print("Searching the Item\n")
    await asyncio.sleep(0.5)
    # Randomly Generate the price for the item

    price = random.randint(1, 100)

    return f"found {item} in the database for price ${price}.00\n"


@function_tool
async def get_shopping_cart(wrapper: RunContextWrapper[UserProfile]) -> str:
    """Gives names of item available in cart"""
    print("Getting Shopping Cart Items\n")
    time.sleep(0.5)
    return f"Items Selected :{", ".join(wrapper.context.shopping_cart)}\n"


@function_tool
async def add_to_cart(wrapper: RunContextWrapper[UserProfile], item: str) -> None:
    """Adds the given item to the cart"""
    print(f"\n Adding {item} to cart\n")
    time.sleep(0.5)
    wrapper.context.shopping_cart.extend(item)


@function_tool
async def purchase_item(wrapper: RunContextWrapper[UserProfile]) -> None:
    """gives names of the items Purchased from cart"""
    print("Purchasing Items in Cart\n")
    print(f"Items Purchased Successfully : {wrapper.context.shopping_cart}\n")

In [5]:
shopping_agent = Agent[UserProfile](
    name="Shopping Assistant",
    instructions="""You are a shopping assistant dedicated to helping the user with their grocery shopping needs.
    Your primary role is to assist in creating a shopping plan that fits within the user's budget.
    Start by getting the user's budget using the tool get_budget.
    Provide recommendations for grocery items based on the budget and user preferences.
    If the user is nearing or exceeding their budget, suggest cheaper alternatives or ask for a revised budget.
    If the user authorizes it, proceed with the purchase using the tool purchase_item.""",
    model="gpt-4o-mini",
    tools=[get_budget, search_for_item, get_shopping_cart, add_to_cart, purchase_item],
)

convo_items: list[TResponseInputItem] = []
print("You are chatting with the Shopping Assistant (type 'exit' to quit) ")
while True:

    user_input = input("You :")
    if "exit" in user_input:
        break

    profile1 = UserProfile(id=231, name="John Doe", shopping_cart=[])

    convo_items.append({"content": user_input, "role": "user"})

    result = await Runner.run(shopping_agent, convo_items, context=profile1)

    print(f"You : {user_input}")
    print(f"Shopping Assistant : {result.final_output}\n")
    convo_items = result.to_input_list()

You are chatting with the Shopping Assistant (type 'exit' to quit) 
Getting Account Balance

You : hi! i am thinking of making russian salad with roasted potatoes today.
Shopping Assistant : Your budget is $100. 

For the Russian salad and roasted potatoes, here are some typical ingredients you'll need:

### Russian Salad Ingredients:
- Mixed vegetables (like carrots, peas, and beans)
- Mayonnaise
- Boiled potatoes
- Boiled eggs
- Salt and pepper
- Optional: Fruits (like apple or pineapple)

### Roasted Potatoes Ingredients:
- Potatoes
- Olive oil or butter
- Herbs (like rosemary or thyme)
- Salt and pepper

Would you like to proceed with adding these items to your cart, or do you have any specific preferences or alternatives in mind?


 Adding Mixed vegetables (carrots, peas, beans) to cart


 Adding Mayonnaise to cart


 Adding Boiled potatoes to cart


 Adding Boiled eggs to cart


 Adding Salt to cart


 Adding Pepper to cart


 Adding Fruits (apple or pineapple) to cart


 Adding 